# M04 Features as Classifiers


## 在 feature 上做 probe

在 synthetic feature activation 上训练一个小分类器，看看哪些恢复出的方向最重要。


In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(31)

feature_names = ["calendar", "citation", "refusal", "tool"]
features = torch.randn(400, 4)
target = 1.4 * features[:, 2] + 0.8 * features[:, 3] - 0.9 * features[:, 1]
labels = (torch.sigmoid(target) > 0.5).float().unsqueeze(1)

probe = torch.nn.Linear(4, 1)
optimizer = torch.optim.Adam(probe.parameters(), lr=0.05)

for step in range(500):
    logits = probe(features)
    loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    probs = torch.sigmoid(probe(features))
    accuracy = ((probs > 0.5).float() == labels).float().mean().item()
    weights = probe.weight.flatten()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(feature_names, weights.tolist(), color=["#4c78a8", "#9ecae9", "#f58518", "#54a24b"])
ax.axhline(0, color="0.75", linewidth=1)
ax.set_title("Probe weights over recovered features")
plt.tight_layout()

print("Final BCE loss:", float(loss.detach()))
print("Accuracy:", round(accuracy, 3))
print("Weights:", dict(zip(feature_names, [round(value, 3) for value in weights.tolist()])))


## 小结

一旦 feature 可以支撑读出任务，feature 的质量就开始变得可衡量。


## 把这本 notebook 变成研究产出

研究工作单 means this notebook is not complete when the cells finish. 请配合 /templates 里的模板，把结果写成可复查的输出。


### 运行前

- 在训练 probe 之前，先定一个 baseline 和一个潜在 confounder。
- 先写清高 accuracy 能证明什么、不能证明什么。
- 提前判断哪个 feature 权重会占主导。


### 运行后

- 解释为什么最大的 probe 权重仍然可能误导你。
- 列出 3 个 confounders 或替代解释。


### 最后交付这些产物

- 1 份包含 baseline 和 confounders 的 probe memo。
- 1 张 feature 权重图。
- 1 个后续标签或任务建议。


## 验收题

- 当 feature 能直接喂给 classifier 时，这说明了什么，又还没有说明什么？
- 如果 probe 准确率很高，你会先检查哪几个 confounders，再决定是否相信这个结果？
- 除了 accuracy，你还会用什么 baseline 或对照来判断这些 feature 真的有读出价值？
- 如果你不能在不重看讲义的情况下独立答出其中至少 2 题，就回去重看文章和你的书面输出。
